In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
import warnings, gc, time
warnings.filterwarnings("ignore")
import numpy  as np
import pandas as pd
from   pathlib import Path
from   sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

BASE = Path("/kaggle/input/competitions/store-sales-time-series-forecasting")
OUT  = Path("/kaggle/working")

t_start = time.time()
print("=" * 60)
print("  STORE SALES — PERFECT SOLUTION")
print("=" * 60)

# ──────────────────────────────────────────────────────────────
# 1. LOAD
# ──────────────────────────────────────────────────────────────
print("\n[1] Loading data...")
train  = pd.read_csv(BASE/"train.csv",           parse_dates=["date"])
test   = pd.read_csv(BASE/"test.csv",            parse_dates=["date"])
stores = pd.read_csv(BASE/"stores.csv")
oil    = pd.read_csv(BASE/"oil.csv",             parse_dates=["date"])
hol    = pd.read_csv(BASE/"holidays_events.csv", parse_dates=["date"])
trans  = pd.read_csv(BASE/"transactions.csv",    parse_dates=["date"])

train["sales"] = train["sales"].clip(lower=0).astype("float32")
print(f"  train={train.shape} | test={test.shape}")
print(f"  Train: {train.date.min().date()} → {train.date.max().date()}")
print(f"  Test:  {test.date.min().date()}  → {test.date.max().date()}")

# ──────────────────────────────────────────────────────────────
# 2. STATIC LOOKUPS
# ──────────────────────────────────────────────────────────────
print("\n[2] Building static lookups...")

# Stores
stores = stores.rename(columns={"type": "store_type"})
le_type = LabelEncoder().fit(stores["store_type"])
stores["store_type_enc"] = le_type.transform(stores["store_type"]).astype("int8")

# Oil — fill every calendar day
oil = (oil.set_index("date")
          .reindex(pd.date_range("2013-01-01", "2017-09-01", freq="D"))
          .rename_axis("date").reset_index())
oil["dcoilwtico"] = oil["dcoilwtico"].interpolate("linear").ffill().bfill()
oil["oil_ma7"]    = oil["dcoilwtico"].rolling(7,  min_periods=1).mean()
oil["oil_ma28"]   = oil["dcoilwtico"].rolling(28, min_periods=1).mean()

# Holidays
nat   = set(hol[hol.locale=="National"]["date"])
xfer  = set(hol[hol.transferred==True]["date"])
reg   = hol[hol.locale=="Regional"].groupby("date")["locale_name"].apply(set).to_dict()
loc   = hol[hol.locale=="Local"].groupby("date")["locale_name"].apply(set).to_dict()
nat_s = sorted(nat)

hol_lkp = pd.DataFrame({"date": pd.date_range("2013-01-01","2017-09-01",freq="D")})
hol_lkp["is_nat"]  = hol_lkp.date.isin(nat).astype("int8")
hol_lkp["is_xfer"] = hol_lkp.date.isin(xfer).astype("int8")
hol_lkp["days_to_hol"] = hol_lkp.date.map(
    lambda d: min(abs((d-h).days) for h in nat_s) if nat_s else 0
).clip(0, 30).astype("int8")

# Transactions
trans = trans.sort_values(["store_nbr","date"])
g = trans.groupby("store_nbr")["transactions"]
trans["tx_ma7"]  = g.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).mean())
trans["tx_ma28"] = g.transform(lambda x: x.shift(1).rolling(28, min_periods=1).mean())

# Encoders
le_fam = LabelEncoder().fit(train["family"])
print(f"  Families: {len(le_fam.classes_)} | Store types: {stores.store_type.nunique()}")

# ──────────────────────────────────────────────────────────────
# 3. CALENDAR + STATIC FEATURE ADDER
# ──────────────────────────────────────────────────────────────
def add_static_features(df):
    df = df.copy()
    d = df["date"]

    # Calendar
    df["year"]          = d.dt.year.astype("int16")
    df["month"]         = d.dt.month.astype("int8")
    df["day"]           = d.dt.day.astype("int8")
    df["dow"]           = d.dt.dayofweek.astype("int8")
    df["doy"]           = d.dt.dayofyear.astype("int16")
    df["woy"]           = d.dt.isocalendar().week.astype("int8").values
    df["quarter"]       = d.dt.quarter.astype("int8")
    df["is_weekend"]    = (d.dt.dayofweek >= 5).astype("int8")
    df["is_month_end"]  = d.dt.is_month_end.astype("int8")
    df["is_month_start"]= d.dt.is_month_start.astype("int8")
    df["is_payday"]     = ((d.dt.day == 15) | d.dt.is_month_end).astype("int8")
    df["trend"]         = (d - pd.Timestamp("2013-01-01")).dt.days.astype("int16")

    # Fourier — weekly seasonality
    for k in [1, 2, 3]:
        df[f"sin_w{k}"] = np.sin(2*np.pi*k*df["dow"]/7).astype("float32")
        df[f"cos_w{k}"] = np.cos(2*np.pi*k*df["dow"]/7).astype("float32")
    # Fourier — annual seasonality
    for k in [1, 2, 3]:
        df[f"sin_y{k}"] = np.sin(2*np.pi*k*df["doy"]/365.25).astype("float32")
        df[f"cos_y{k}"] = np.cos(2*np.pi*k*df["doy"]/365.25).astype("float32")

    # Store
    df = df.merge(
        stores[["store_nbr","store_type_enc","cluster","city","state"]],
        on="store_nbr", how="left")

    # Holidays
    df = df.merge(hol_lkp, on="date", how="left")
    df["is_reg"] = df.apply(
        lambda r: int(r["state"] in reg.get(r["date"], set())), axis=1).astype("int8")
    df["is_loc"] = df.apply(
        lambda r: int(r["city"]  in loc.get(r["date"], set())), axis=1).astype("int8")
    df["is_hol"] = (df.is_nat | df.is_reg | df.is_loc).astype("int8")
    df.drop(columns=["city","state"], inplace=True)

    # Oil
    df = df.merge(oil[["date","dcoilwtico","oil_ma7","oil_ma28"]], on="date", how="left")

    # Transactions
    df = df.merge(trans[["date","store_nbr","transactions","tx_ma7","tx_ma28"]],
                  on=["date","store_nbr"], how="left")

    # Encode
    df["fam_enc"]      = le_fam.transform(df["family"]).astype("int8")
    df["store_fam"]    = (df["store_nbr"].astype("int16")*100 +
                           df["fam_enc"].astype("int16")).astype("int16")
    return df

# ──────────────────────────────────────────────────────────────
# 4. LAG FEATURES — TWO SEPARATE PIPELINES
#
#    PIPELINE A (TRAIN): groupby().shift() on train-only
#    PIPELINE B (TEST):  date-shift merge from train → ZERO NaN
#
#    LAGS USED:
#    • 16..56 : recent history (safe: lag_16 of Aug31 = Aug15 ✓)
#    • 91,182 : medium-term
#    • 364,365,366 : same time last year
#
#    ROLLING WINDOWS:
#    • For TRAIN: rolling(w).mean() shifted by 1
#    • For TEST:  fixed window ending Aug 15 per (store,family)
# ──────────────────────────────────────────────────────────────
print("\n[3] Computing lag features...")

LAGS    = [16, 21, 28, 35, 42, 56, 91, 182, 364, 365, 366]
WINDOWS = [7, 14, 28, 56]

# ── PIPELINE A: TRAIN LAGS ──────────────────────────────────
print("  A) Training lags via groupby().shift()...")
t0 = time.time()
train_s = train.sort_values(["store_nbr","family","date"]).reset_index(drop=True)
grp = train_s.groupby(["store_nbr","family"])["sales"]

for lag in LAGS:
    train_s[f"lag_{lag}"] = grp.shift(lag).astype("float32")

for w in WINDOWS:
    train_s[f"roll_mean_{w}"] = grp.shift(1).transform(
        lambda x: x.rolling(w, min_periods=1).mean()).astype("float32")
    train_s[f"roll_std_{w}"]  = grp.shift(1).transform(
        lambda x: x.rolling(w, min_periods=1).std().fillna(0)).astype("float32")
    train_s[f"roll_max_{w}"]  = grp.shift(1).transform(
        lambda x: x.rolling(w, min_periods=1).max()).astype("float32")

# Last-year average (3 lags averaged)
train_s["lag_ly_avg"] = (
    train_s["lag_364"] + train_s["lag_365"] + train_s["lag_366"]
).astype("float32") / 3

print(f"  Train lags done in {time.time()-t0:.0f}s | shape: {train_s.shape}")

# ── PIPELINE B: TEST LAGS via date-shift merge ──────────────
print("  B) Test lags via date-shift merge (guaranteed zero NaN)...")
t0 = time.time()

# Source: all training sales indexed by (date, store_nbr, family)
src = train[["date","store_nbr","family","sales"]].copy()

test_s = test.sort_values(["store_nbr","family","date"]).reset_index(drop=True)

for lag in LAGS:
    lsrc = src.copy()
    lsrc["date"] = lsrc["date"] + pd.DateOffset(days=lag)
    lsrc = lsrc.rename(columns={"sales": f"lag_{lag}"})
    test_s = test_s.merge(
        lsrc[["date","store_nbr","family",f"lag_{lag}"]],
        on=["date","store_nbr","family"], how="left")
    n = test_s[f"lag_{lag}"].isna().sum()
    if n > 0:
        print(f"    WARNING: lag_{lag} has {n} NaN!")
    else:
        print(f"    lag_{lag}: 0 NaN ✓")

# Rolling for test: fixed window ending on last train date (Aug 15)
# This is correct because test has NO sales — we use the most recent
# available window from training data
print("  B2) Rolling windows for test (fixed window from train end)...")
LAST_TRAIN = pd.Timestamp("2017-08-15")

for w in WINDOWS:
    col_mean = f"roll_mean_{w}"
    col_std  = f"roll_std_{w}"
    col_max  = f"roll_max_{w}"

    # Compute per (store, family): rolling stat of last w days of train
    roll_stats = (
        train[train.date > LAST_TRAIN - pd.Timedelta(days=w+1)]
        .groupby(["store_nbr","family"])["sales"]
        .agg(
            **{col_mean: "mean",
               col_std:  "std",
               col_max:  "max"}
        )
        .reset_index()
    )
    roll_stats[col_std]  = roll_stats[col_std].fillna(0).astype("float32")
    roll_stats[col_mean] = roll_stats[col_mean].astype("float32")
    roll_stats[col_max]  = roll_stats[col_max].astype("float32")
    test_s = test_s.merge(roll_stats, on=["store_nbr","family"], how="left")

# Last-year average for test
test_s["lag_ly_avg"] = (
    test_s["lag_364"] + test_s["lag_365"] + test_s["lag_366"]
).astype("float32") / 3

print(f"  Test lags done in {time.time()-t0:.0f}s | shape: {test_s.shape}")
gc.collect()

# ──────────────────────────────────────────────────────────────
# 5. ADD STATIC FEATURES TO BOTH
# ──────────────────────────────────────────────────────────────
print("\n[4] Adding static features...")
train_s["family"] = train_s["family"].astype(str)
test_s["family"]  = test_s["family"].astype(str)
train_feat = add_static_features(train_s)
test_feat  = add_static_features(test_s)
print(f"  train_feat: {train_feat.shape} | test_feat: {test_feat.shape}")
gc.collect()

# ──────────────────────────────────────────────────────────────
# 6. FEATURE COLUMNS
# ──────────────────────────────────────────────────────────────
SKIP = {"id","date","family","sales","store_type",
        "city","state","transactions"}
FEATURES = [c for c in train_feat.columns
            if c not in SKIP and c in test_feat.columns]

# Fill NaN (only early training rows where history doesn't exist)
train_feat[FEATURES] = train_feat[FEATURES].fillna(0)
test_feat[FEATURES]  = test_feat[FEATURES].fillna(0)

print(f"\n  Feature count: {len(FEATURES)}")
print(f"  Features:\n  {FEATURES}")

# ──────────────────────────────────────────────────────────────
# 7. LOG TRANSFORM TARGET
# ──────────────────────────────────────────────────────────────
train_feat["log1p_sales"] = np.log1p(train_feat["sales"]).astype("float32")

# ──────────────────────────────────────────────────────────────
# 8. TRAIN / VALIDATION SPLIT
#    Val = last 16 days of train (same length as test window)
#    Drop rows with no 182-day history (too early in series)
# ──────────────────────────────────────────────────────────────
VAL_CUT   = pd.Timestamp("2017-07-31")
train_use = train_feat.dropna(subset=["lag_182"])

X_tr = train_use[train_use.date <  VAL_CUT][FEATURES]
y_tr = train_use[train_use.date <  VAL_CUT]["log1p_sales"]
X_va = train_use[train_use.date >= VAL_CUT][FEATURES]
y_va = train_use[train_use.date >= VAL_CUT]["log1p_sales"]
X_te = test_feat[FEATURES]

print(f"\n  X_tr={X_tr.shape} | X_va={X_va.shape} | X_te={X_te.shape}")
del train_s, train_feat, train_use; gc.collect()

# ──────────────────────────────────────────────────────────────
# 9. LIGHTGBM — 3 SEEDS
# ──────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("[5] Training LightGBM — 3-seed ensemble")
print("="*60)

PARAMS = dict(
    objective         = "regression_l1",  # MAE — robust to zero-sales outliers
    metric            = "rmse",
    boosting_type     = "gbdt",
    num_leaves        = 255,
    learning_rate     = 0.05,
    feature_fraction  = 0.8,
    bagging_fraction  = 0.8,
    bagging_freq      = 1,
    min_child_samples = 20,
    reg_alpha         = 0.05,
    reg_lambda        = 0.5,
    n_jobs            = -1,
    verbose           = -1,
)

dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
dvalid = lgb.Dataset(X_va, label=y_va, reference=dtrain, free_raw_data=False)

models, scores = [], []
for seed in [42, 123, 777]:
    print(f"\n  ── seed={seed} ──")
    PARAMS["seed"] = seed
    t0 = time.time()
    m = lgb.train(
        PARAMS, dtrain,
        num_boost_round = 3000,
        valid_sets      = [dvalid],
        callbacks       = [lgb.log_evaluation(100),
                           lgb.early_stopping(50, verbose=True)],
    )
    vp    = np.expm1(m.predict(X_va, num_iteration=m.best_iteration)).clip(0)
    vt    = np.expm1(y_va.values).clip(0)
    rmsle = np.sqrt(np.mean((np.log1p(vp) - np.log1p(vt))**2))
    print(f"  Val RMSLE={rmsle:.5f} | iter={m.best_iteration} | {time.time()-t0:.0f}s")
    models.append(m)
    scores.append(rmsle)

print(f"\n  Scores: {[f'{s:.5f}' for s in scores]}")
print(f"  Mean:   {np.mean(scores):.5f}")

# Feature importance
fi = pd.Series(models[0].feature_importance("gain"), index=FEATURES)
print("\n  Top 20 features:")
print(fi.sort_values(ascending=False).head(20).to_string())

# ──────────────────────────────────────────────────────────────
# 10. PREDICT
# ──────────────────────────────────────────────────────────────
print("\n[6] Predicting...")
inv_w   = 1.0 / np.array(scores)
weights = inv_w / inv_w.sum()
log_p   = sum(w * m.predict(X_te, num_iteration=m.best_iteration)
              for m, w in zip(models, weights))
preds   = np.expm1(log_p).clip(min=0)
print(f"  min={preds.min():.2f} | mean={preds.mean():.2f} | max={preds.max():.2f}")

# ──────────────────────────────────────────────────────────────
# 11. SUBMISSION
# ──────────────────────────────────────────────────────────────
print("\n[7] Building submission...")
test_s["sales"] = preds
sample = pd.read_csv(BASE/"sample_submission.csv")
sub    = sample[["id"]].merge(test_s[["id","sales"]], on="id", how="left")
sub["sales"] = sub["sales"].fillna(0).clip(lower=0)
sub.to_csv(OUT/"submission.csv", index=False)

print("\n" + "="*60)
print(f"  ✅  submission.csv saved! ({len(sub):,} rows)")
print(f"  Sales: mean={sub.sales.mean():.2f} | max={sub.sales.max():.2f}")
print(f"  Val RMSLE: {[f'{s:.5f}' for s in scores]}")
print(f"  Expected LB: ~0.376 – 0.382")
print(f"  Total time: {(time.time()-t_start)/60:.1f} min")
print("="*60)
print("  → Submit Prediction → upload submission.csv")

  STORE SALES — PERFECT SOLUTION

[1] Loading data...
  train=(3000888, 6) | test=(28512, 5)
  Train: 2013-01-01 → 2017-08-15
  Test:  2017-08-16  → 2017-08-31

[2] Building static lookups...
  Families: 33 | Store types: 5

[3] Computing lag features...
  A) Training lags via groupby().shift()...
  Train lags done in 5s | shape: (3000888, 30)
  B) Test lags via date-shift merge (guaranteed zero NaN)...
    lag_16: 0 NaN ✓
    lag_21: 0 NaN ✓
    lag_28: 0 NaN ✓
    lag_35: 0 NaN ✓
    lag_42: 0 NaN ✓
    lag_56: 0 NaN ✓
    lag_91: 0 NaN ✓
    lag_182: 0 NaN ✓
    lag_364: 0 NaN ✓
    lag_365: 0 NaN ✓
    lag_366: 0 NaN ✓
  B2) Rolling windows for test (fixed window from train end)...
  Test lags done in 10s | shape: (28512, 29)

[4] Adding static features...
  train_feat: (3000888, 70) | test_feat: (28512, 69)

  Feature count: 65
  Features:
  ['store_nbr', 'onpromotion', 'lag_16', 'lag_21', 'lag_28', 'lag_35', 'lag_42', 'lag_56', 'lag_91', 'lag_182', 'lag_364', 'lag_365', 'lag_366'